## Build a RAG agent with LangChain

### Overview

RAG - Retrieval Augmented Generation

This tutorial will show how to build a simple Q&A application over an unstructured text data source. We will demonstrate:

1. A RAG agent that executes searches with a simple tool. This is a good general-purpose implementation.
2. A two-step RAG chain that uses just a single LLM call per query. This is a fast and effective method for simple queries.

### Concepts

We will cover the following concepts:

- **Indexing:** a pipeline for ingesting data from a source and indexing it. This usually happens in a separate process.
- **Retrieval and generation:** the actual RAG process, which takes the user query at run time and retrieves the relevant data from the index, then passes that to the model.

Once we’ve indexed our data, we will use an agent as our orchestration framework to implement the retrieval and generation steps.

#### Preview

In this guide we’ll build an app that answers questions about the website’s content. The specific website we will use is the LLM Powered Autonomous Agents blog post by Lilian Weng, which allows us to ask questions about the contents of the post.

### Setup

#### Dependencies


```bash
pip install langchain langchain-text-splitters langchain-community bs4
```

#### LangSmith

Using environment variables

```bash
export LANGSMITH_TRACING="true"
export LANGSMITH_API_KEY="..."
```

Using getpass

```bash
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()
```

Using python-dotenv

(Make sure environment variables are assigned in .env file)

```bash
pip install python-dotenv
```

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

#### Components

We will need to select three components from LangChain’s suite of integrations.

- Select a chat model
- Select an embeddings model
- Select a vector store

Select a chat model:

```bash
pip install -U "langchain[openai]"
```

In [2]:
import os
import getpass

from langchain.chat_models import init_chat_model

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

model = init_chat_model("gpt-4.1")

Select an embeddings model:

```bash
pip install -U "langchain-openai"
```

In [3]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

Select a vector store:

```bash
pip install -U "langchain-core"
```

In [4]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

### Indexing

Indexing commonly works as follows:

1. **Load:** First we need to load our data. This is done with Document Loaders.
2. **Split:** Text splitters break large Documents into smaller chunks. This is useful both for indexing data and passing it into a model, as large chunks are harder to search over and won’t fit in a model’s finite context window.
3. **Store:** We need somewhere to store and index our splits, so that they can be searched over later. This is often done using a VectorStore and Embeddings model.

#### Loading documents

We can use DocumentLoaders for this, which are objects that load in data from a source and return a list of Document objects. In this case we’ll use the WebBaseLoader, which uses urllib to load HTML from web URLs and BeautifulSoup to parse it to text



In [5]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Total characters: 43047


In [6]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


#### Splitting documents

Let's use a RecursiveCharacterTextSplitter, which will recursively split the document using common separators like new lines until each chunk is the appropriate size. This is the recommended text splitter for generic text use cases

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # chunk size (characters)
    chunk_overlap=200,      # chunk overlap (characters)
    add_start_index=True,   # track index in original document
)

all_splits = text_splitter.split_documents(docs)

print(f"Split blogpost into {len(all_splits)} chunks")

Split blogpost into 63 chunks


#### Storing documents

Let's store all splits in the vector store

In [ ]:
# Add the split documents to the vector store.
# This operation indexes the content of each document and stores its vector representation,
# making it searchable.
document_ids = vector_store.add_documents(documents=all_splits)

# Print the first three document IDs returned after adding the documents.
# These IDs can be used to reference the documents within the vector store.
print(document_ids[:3])

['47a476a7-f11b-4095-8f9a-f92efefa954c', 'f08098f0-926a-4aed-a6ad-e1e093c5b064', '6a8b5bf9-98a6-41bb-b5be-7603733e1f56']
